In [ ]:
import os
import zarr
import numpy as np
from transcription_pipeline import time_filter
from glob import glob
import shutil


def process_embryo_time_filter(embryo_rel_path, dataset_folder, time_chunk=50, temporal_window=21):
    """
    Rechunk and apply temporal mean filtering to all collated_dataset*.zarr files in the embryo folder.
    Skips rechunking or filtering if output files already exist.

    Parameters:
        embryo_rel_path (str): Path relative to dataset_folder for the embryo
        dataset_folder (str): Base dataset folder
        time_chunk (int): Chunk size along time axis for rechunking
        temporal_window (int): Window size for temporal mean filter
    """
    embryo_full_path = os.path.join(dataset_folder, embryo_rel_path, "collated_dataset")

    # Find all collated_dataset*.zarr files
    zarr_files = glob(os.path.join(embryo_full_path, "collated_dataset*.zarr"))

    if not zarr_files:
        print(f"\n[WARNING] No collated_dataset*.zarr files found in:\n  {embryo_full_path}\n")
        return

    print(f"\n{'=' * 70}")
    print(f"PROCESSING TIME FILTER")
    print(f"{'=' * 70}")
    print(f"Embryo path: {embryo_rel_path}")
    print(f"Found {len(zarr_files)} zarr file(s) to process\n")

    for z_file in zarr_files:
        base_name = os.path.basename(z_file)
        
        # Skip if this is already a rechunked or filtered file
        if "_rechunk" in base_name or "_time_filtered" in base_name:
            continue
            
        rechunk_store = os.path.join(
            embryo_full_path,
            base_name.replace(".zarr", "_rechunk.zarr")
        )
        filtered_store = os.path.join(
            embryo_full_path,
            base_name.replace(".zarr", "_time_filtered.zarr")
        )

        print(f"\n{'-' * 70}")
        print(f"Processing: {base_name}")
        print(f"{'-' * 70}")

        # Skip if already filtered
        if os.path.exists(filtered_store):
            print(f"  ✓ Filtered file already exists - skipping all processing")
            continue

        # ------------------------------
        # Rechunk
        # ------------------------------
        if not os.path.exists(rechunk_store):
            print(f"  → Rechunking data (chunk size: {time_chunk})...")
            os.makedirs(os.path.dirname(rechunk_store), exist_ok=True)
            img = zarr.open(z_file, mode='r')
            chunks = (time_chunk, *img.shape[1:])
            dst_rechunk = zarr.open_array(
                store=rechunk_store,
                mode="w",
                shape=img.shape,
                chunks=chunks,
                dtype=np.float32,
            )
            dst_rechunk[:] = img[:].astype(np.float32)
            os.sync()
            print(f"  ✓ Rechunking complete")
        else:
            print(f"  ✓ Rechunked file already exists - reusing")

        # ------------------------------
        # Temporal mean filter
        # ------------------------------
        print(f"  → Applying temporal mean filter (window: {temporal_window})...")
        os.makedirs(os.path.dirname(filtered_store), exist_ok=True)
        dst_filtered = zarr.open_array(
            store=filtered_store,
            mode="w",
            shape=zarr.open(rechunk_store, mode='r').shape,
            chunks=(time_chunk, *zarr.open(rechunk_store, mode='r').shape[1:]),
            dtype=np.float32,
        )

        time_filter.temporal_filter_zarr(
            zarr.open(rechunk_store, mode="r"),
            temporal_window,
            mode="mean",
            out=dst_filtered
        )

        print(f"  ✓ Time filtering complete\n")

    print(f"{'=' * 70}")
    print(f"PROCESSING COMPLETE")
    print(f"{'=' * 70}\n")


def clean_rechunk_time_filtering(embryo_rel_path, dataset_folder, overwrite_original=False):
    """
    Verify that time filtering was successful, delete temporary rechunk files if successful,
    and rechunk the time_filtered zarr back to original chunking for both channels.

    Parameters:
        embryo_rel_path (str): Path relative to dataset_folder for the embryo
        dataset_folder (str): Base dataset folder
        overwrite_original (bool): If True, overwrite the original time_filtered zarr with the rechunked version
    """
    embryo_full_path = os.path.join(dataset_folder, embryo_rel_path, "collated_dataset")

    # Find all original collated_dataset*.zarr files (not rechunk or time_filtered)
    zarr_files = [f for f in glob(os.path.join(embryo_full_path, "collated_dataset*.zarr"))
                  if "_rechunk" not in f and "_time_filtered" not in f]

    if not zarr_files:
        print(f"\n[WARNING] No original collated_dataset*.zarr files found in:\n  {embryo_full_path}\n")
        return

    print(f"\n{'=' * 70}")
    print(f"CLEANING AND RECHUNKING FILTERED DATA")
    print(f"{'=' * 70}")
    print(f"Embryo path: {embryo_rel_path}")
    print(f"Overwrite mode: {'ENABLED' if overwrite_original else 'DISABLED'}")
    print(f"Found {len(zarr_files)} file(s) to process\n")

    for z_file in zarr_files:
        base_name = os.path.basename(z_file)
        rechunk_store = os.path.join(
            embryo_full_path,
            base_name.replace(".zarr", "_rechunk.zarr")
        )
        filtered_store = os.path.join(
            embryo_full_path,
            base_name.replace(".zarr", "_time_filtered.zarr")
        )
        final_store = os.path.join(
            embryo_full_path,
            base_name.replace(".zarr", "_time_filtered_rechunked.zarr")
        )

        print(f"\n{'-' * 70}")
        print(f"Processing: {base_name}")
        print(f"{'-' * 70}")

        # Check if filtered file exists
        if not os.path.exists(filtered_store):
            print(f"  ✗ Time filtered file not found - skipping")
            continue

        # Check if already processed (final rechunked version exists or overwrite already done)
        if not overwrite_original and os.path.exists(final_store):
            print(f"  ✓ Final rechunked file already exists - skipping")
            # Still clean up rechunk file if it exists
            if os.path.exists(rechunk_store):
                print(f"  → Deleting temporary rechunk file...")
                shutil.rmtree(rechunk_store)
                print(f"  ✓ Temporary files removed")
            continue

        # Verify the filtered file is valid
        try:
            original = zarr.open(z_file, mode='r')
            filtered = zarr.open(filtered_store, mode='r')

            # Check shapes match
            if original.shape != filtered.shape:
                print(f"  ✗ Shape mismatch: original {original.shape} vs filtered {filtered.shape}")
                continue

            # Check data is not all zeros or NaN
            sample_data = filtered[0:min(5, filtered.shape[0])]
            if np.all(sample_data == 0) or np.all(np.isnan(sample_data)):
                print(f"  ✗ Invalid filtered data detected (all zeros or NaN)")
                continue

            print(f"  ✓ Verification successful")

            # Delete temporary rechunk file
            if os.path.exists(rechunk_store):
                print(f"  → Deleting temporary rechunk file...")
                shutil.rmtree(rechunk_store)
                print(f"  ✓ Temporary files removed")

            # Rechunk filtered file back to original chunking
            if overwrite_original:
                temp_store = os.path.join(
                    embryo_full_path,
                    base_name.replace(".zarr", "_time_filtered_temp.zarr")
                )

                print(f"  → Rechunking to original chunks (will overwrite)...")
                os.makedirs(os.path.dirname(temp_store), exist_ok=True)

                dst_temp = zarr.open_array(
                    store=temp_store,
                    mode="w",
                    shape=filtered.shape,
                    chunks=original.chunks,
                    dtype=filtered.dtype,
                )
                dst_temp[:] = filtered[:]
                os.sync()

                print(f"  → Replacing original with rechunked version...")
                shutil.rmtree(filtered_store)
                os.rename(temp_store, filtered_store)
                print(f"  ✓ Overwrite complete")
            else:
                if not os.path.exists(final_store):
                    print(f"  → Rechunking to original chunks...")
                    os.makedirs(os.path.dirname(final_store), exist_ok=True)

                    dst_final = zarr.open_array(
                        store=final_store,
                        mode="w",
                        shape=filtered.shape,
                        chunks=original.chunks,
                        dtype=filtered.dtype,
                    )
                    dst_final[:] = filtered[:]
                    os.sync()
                    print(f"  ✓ Rechunking complete")
                else:
                    print(f"  ✓ Final rechunked file already exists")

        except Exception as e:
            print(f"  ✗ Error: {type(e).__name__}: {str(e)}")
            continue

    print(f"\n{'=' * 70}")
    print(f"CLEANING COMPLETE")
    print(f"{'=' * 70}\n")


In [42]:
dataset_folder = '/mnt/Data6/Enze/transcription_pipeline'

embryo_path = 'Data/2025-10-18-Enze/1DgVw_MCP-mSG_His-RFP004'

process_embryo_time_filter(embryo_path, dataset_folder)
clean_rechunk_time_filtering(embryo_path, dataset_folder, overwrite_original=True)


PROCESSING TIME FILTER
Embryo path: Data/2025-10-18-Enze/1DgVw_MCP-mSG_His-RFP004
Found 3 zarr file(s) to process


----------------------------------------------------------------------
Processing: collated_dataset_ch02.zarr
----------------------------------------------------------------------
  → Rechunking data (chunk size: 50)...
  ✓ Rechunking complete
  → Applying temporal mean filter (window: 21)...


Mean filtering: 100%|██████████| 707/707 [00:04<00:00, 147.09it/s]


  ✓ Time filtering complete


----------------------------------------------------------------------
Processing: collated_dataset_ch01.zarr
----------------------------------------------------------------------
  → Rechunking data (chunk size: 50)...
  ✓ Rechunking complete
  → Applying temporal mean filter (window: 21)...


Mean filtering: 100%|██████████| 707/707 [00:03<00:00, 190.47it/s]


  ✓ Time filtering complete


----------------------------------------------------------------------
Processing: collated_dataset_ch00.zarr
----------------------------------------------------------------------
  → Rechunking data (chunk size: 50)...
  ✓ Rechunking complete
  → Applying temporal mean filter (window: 21)...


Mean filtering: 100%|██████████| 707/707 [00:04<00:00, 152.43it/s]


  ✓ Time filtering complete

PROCESSING COMPLETE


CLEANING AND RECHUNKING FILTERED DATA
Embryo path: Data/2025-10-18-Enze/1DgVw_MCP-mSG_His-RFP004
Overwrite mode: ENABLED
Found 3 file(s) to process


----------------------------------------------------------------------
Processing: collated_dataset_ch02.zarr
----------------------------------------------------------------------
  ✓ Verification successful
  → Deleting temporary rechunk file...
  ✓ Temporary files removed
  → Rechunking to original chunks (will overwrite)...
  → Replacing original with rechunked version...
  ✓ Overwrite complete

----------------------------------------------------------------------
Processing: collated_dataset_ch01.zarr
----------------------------------------------------------------------
  ✓ Verification successful
  → Deleting temporary rechunk file...
  ✓ Temporary files removed
  → Rechunking to original chunks (will overwrite)...
  → Replacing original with rechunked version...
  ✓ Overwrite com

## Code from Nick

In [ ]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '002'

embryo_list = {
    '001': [
        # 'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',

    ],

    '002': [
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],

    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03',
        # 'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ]
}

In [ ]:
# Run the filtering function on each embryo
for key in embryo_list:
    for embryo in embryo_list[key]:
        process_embryo_time_filter(embryo, dataset_folder)
        clean_rechunk_time_filtering(embryo, dataset_folder, overwrite_original=True)

In [3]:
# Run the filtering function on single embryo
process_embryo_time_filter(embryo_list[key][0], dataset_folder)


PROCESSING TIME FILTER
Embryo path: test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03
Found 2 zarr file(s) to process


----------------------------------------------------------------------
Processing: collated_dataset_ch01.zarr
----------------------------------------------------------------------
  → Rechunking data (chunk size: 50)...
  ✓ Rechunking complete
  → Applying temporal mean filter (window: 21)...


Mean filtering: 100%|██████████| 399/399 [00:15<00:00, 25.10it/s]


  ✓ Time filtering complete


----------------------------------------------------------------------
Processing: collated_dataset_ch00.zarr
----------------------------------------------------------------------
  → Rechunking data (chunk size: 50)...
  ✓ Rechunking complete
  → Applying temporal mean filter (window: 21)...


Mean filtering: 100%|██████████| 399/399 [00:17<00:00, 23.04it/s]

  ✓ Time filtering complete

PROCESSING COMPLETE



In [4]:
# Run the clean and rechunk function on each embryo
clean_rechunk_time_filtering(embryo_list[key][0], dataset_folder, overwrite_original=True)


CLEANING AND RECHUNKING FILTERED DATA
Embryo path: test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03
Overwrite mode: ENABLED
Found 2 file(s) to process


----------------------------------------------------------------------
Processing: collated_dataset_ch01.zarr
----------------------------------------------------------------------
  ✓ Verification successful
  → Deleting temporary rechunk file...
  ✓ Temporary files removed
  → Rechunking to original chunks (will overwrite)...
  → Replacing original with rechunked version...
  ✓ Overwrite complete

----------------------------------------------------------------------
Processing: collated_dataset_ch00.zarr
----------------------------------------------------------------------
  ✓ Verification successful
  → Deleting temporary rechunk file...
  ✓ Temporary files removed
  → Rechunking to original chunks (will overwrite)...
  → Replacing original with rechunked version...
  ✓ Overwrite complete

CLEANING COMPLET